# Unified Namespace Basics

This notebook demonstrates the general unified namespace feature in DataX: ordinary variables created in one language become available in the next language cell.

The examples intentionally use scalars, lists, dictionaries, and simple vectors. Apache Arrow and tabular data transfer are covered separately in `04_arrow_variable_sharing.ipynb`.

## 1. Create User Variables in Python

These names are normal Python globals. When the next cell switches to R, DataX synchronizes the shareable values automatically.

In [1]:
project_name = "harbor-retail"
months = ["Jan", "Feb", "Mar", "Apr"]
revenue = [125_000, 142_000, 158_000, 181_000]
costs = [79_000, 86_000, 94_000, 104_000]

assumptions = {
    "currency": "USD",
    "growth_target": 0.12,
    "segments": ["online", "store", "partner"],
}

print(project_name)
print("Revenue:", revenue)
print("Assumptions:", assumptions)

harbor-retail
Revenue: [125000, 142000, 158000, 181000]
Assumptions: {'currency': 'USD', 'growth_target': 0.12, 'segments': ['online', 'store', 'partner']}


## 2. Inspect the Sharing State

The `%sharing status` magic reports whether automatic namespace sharing is enabled for the current kernel session.

In [2]:
%sharing status


=== Namespace Sharing Status ===
Auto-sync: enabled
Change tracking: enabled

Shared Variables: 0
Total size: 0 KB

Format breakdown:

Language breakdown:

Sync statistics:
  Total syncs: 0
  Successful: 0
  Failed: 0
  Average time: 0 ms

Unified FFI v2 state:
  python baseline: 80, reserved skips: 0, equal skips: 0
  r baseline: 0, reserved skips: 0, equal skips: 0
  javascript baseline: 0, reserved skips: 0, equal skips: 0
  last sync: none

History Statistics:
  Total entries: 1
  Full snapshots: 1
  Delta snapshots: 0
  Total size: 0.230469 KB
  Time span: 2026-08-09 07:06:44 to 2026-08-09 07:06:44
  Compression: ~100% space savings

Snapshots: 2
History entries: 0



## 3. Use Python Variables Directly in R

R receives the Python-created lists as ordinary R vectors and can create new variables for Python to read later.

In [3]:
%%R
cat("Project from Python:", project_name, "\n")
cat("Months:", paste(months, collapse = ", "), "\n")

profit <- revenue - costs
margin_pct <- round(profit / revenue * 100, 1)
best_month <- months[which.max(profit)]
average_margin <- round(mean(margin_pct), 1)

cat("Profit:", paste(profit, collapse = ", "), "\n")
cat("Margin %:", paste(margin_pct, collapse = ", "), "\n")

Project from Python: harbor-retail 
Months: Jan, Feb, Mar, Apr 
Profit: 46000, 56000, 64000, 77000 
Margin %: 36.8, 39.4, 40.5, 42.5 

## 4. Read R Results Back in Python

The variables `profit`, `margin_pct`, `best_month`, and `average_margin` were created in R. The next Python cell reads them without loading files or calling a conversion helper.

In [4]:
for month, revenue_value, profit_value, margin_value in zip(months, revenue, profit, margin_pct):
    print(f"{month}: revenue=${revenue_value:,}, profit=${int(profit_value):,}, margin={margin_value:.1f}%")

print(f"\nBest month from R: {best_month}")
print(f"Average margin from R: {average_margin:.1f}%")

Jan: revenue=$125,000, profit=$46,000, margin=36.8%
Feb: revenue=$142,000, profit=$56,000, margin=39.4%
Mar: revenue=$158,000, profit=$64,000, margin=40.5%
Apr: revenue=$181,000, profit=$77,000, margin=42.5%

Best month from R: Apr
Average margin from R: 39.8%


In [5]:
summary_payload = {
    "project": project_name,
    "best_month": str(best_month),
    "average_margin_pct": float(average_margin),
    "profitable_months": [month for month, value in zip(months, profit) if value > 40_000],
}

summary_payload

{'project': 'harbor-retail',
 'best_month': 'Apr',
 'average_margin_pct': 39.8,
 'profitable_months': ['Jan', 'Feb', 'Mar', 'Apr']}

## 5. Built-In Names Stay Protected

Namespace sharing should not overwrite target-language built-ins. This small check creates Python variables named like common R functions, then verifies that R still sees its own functions.

In [6]:
c = [10, 20, 30]
t = "python value named t"

print("Python c:", c)
print("Python t:", t)

Python c: [10, 20, 30]
Python t: python value named t


In [7]:
%%R
cat("R c is still a function:", is.function(c), "\n")
cat("R t is still a function:", is.function(t), "\n")
cat("Combining with R's c():", paste(c(1, 2, 3), collapse = ", "), "\n")

R c is still a function: TRUE 
R t is still a function: FALSE 
Combining with R's c(): 1, 2, 3 

## 6. Continue the Workflow in JavaScript

JavaScript receives the Python and R variables too. Here it builds a small report object and text summary for Python to consume.

In [8]:
%%js
delete globalThis.namespace_report;

globalThis.namespace_report_text = [
  `Project: ${project_name}`,
  `Months analyzed: ${months.length}`,
  `Best month: ${best_month}`,
  `Average margin: ${average_margin}% (${average_margin >= 35 ? "healthy" : "watch"})`,
  `Payload keys: ${Object.keys(summary_payload).sort().join(", ")}`
].join("\n");

console.log(namespace_report_text);

Project: harbor-retail
Months analyzed: 4
Best month: Apr
Average margin: 39.8% (healthy)
Payload keys: average_margin_pct, best_month, profitable_months, project


In [9]:
print(namespace_report_text)

Project: harbor-retail
Months analyzed: 4
Best month: Apr
Average margin: 39.8% (healthy)
Payload keys: average_margin_pct, best_month, profitable_months, project


## Summary

| Step | Language | Namespace feature shown |
|------|----------|-------------------------|
| Create values | Python | Scalars, lists, and dictionaries enter the shared namespace |
| Analyze values | R | Python lists are available as R vectors |
| Read results | Python | R-created values return automatically |
| Protect built-ins | Python -> R | Reserved target-language names are not overwritten |
| Build report | JavaScript | JS reads shared values and publishes new ones |

No Apache Arrow tables, files, or manual serialization were used in this notebook.